#직업안정성

(1안) 현재 직업의 안정성에 가장 가까운 유형을 선택해주세요.

- 안정적 직업군
  공무원, 교사, 군인, 경찰/소방, 전문직, 공공기관/공기업 종사자 등

- 일반 직장인·기술직
  일반 사무직, IT/개발직, 기술직, 연구직, 생산관리직, 일반 회사원 등

- 소득 변동 가능 직업군
  자영업, 프리랜서, 영업·판매직, 배달/서비스직, 시간강사, 무직, 구직중 등

(2안) 현재 소득 안정성에 가장 가까운 항목을 선택해주세요.

1. 공공·전문직 중심
   공무원, 교사, 군인, 경찰/소방, 전문직 등

2. 일반 직장·기술직 중심
   사무직, IT/개발직, 기술직, 연구직, 생산관리직 등

3. 자영업·프리랜서 중심
   자영업, 프리랜서, 영업·판매직, 서비스직, 무직/구직중 등

In [64]:
JOB_STABILITY_UI_MAP = {
    "안정적 직업군": {
        "category": "HIGH",
        "score": 1.0
    },
    "일반 직장인·기술직": {
        "category": "MID",
        "score": 0.5
    },
    "소득 변동 가능 직업군": {
        "category": "LOW",
        "score": 0.0
    }
}

In [65]:
def get_job_stability_from_ui(selected_job_type):
    mapping = {
        "안정적 직업군": ("HIGH", 1.0),
        "일반 직장인·기술직": ("MID", 0.5),
        "소득 변동 가능 직업군": ("LOW", 0.0),
    }

    category, score = mapping.get(selected_job_type, ("MID", 0.5))

    return category, score

In [66]:
selected_job_type = "안정적 직업군"

job_stability_category, job_stability_score = get_job_stability_from_ui(selected_job_type)

print(job_stability_category)
print(job_stability_score)

HIGH
1.0


In [67]:
selected_job_type = "일반 직장인·기술직"

job_stability_category, job_stability_score = get_job_stability_from_ui(selected_job_type)

print(job_stability_category)
print(job_stability_score)

MID
0.5


In [68]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 체크박스 옵션 (JOB_STABILITY_UI_MAP에서 가져옴)
job_stability_options = {
    "안정적 직업군": "안정적 직업군",
    "일반 직장인·기술직": "일반 직장인·기술직",
    "소득 변동 가능 직업군": "소득 변동 가능 직업군"
}

# 체크박스 생성
# 여기서는 단일 선택을 가정하므로 라디오 버튼과 유사하게 구현합니다.
# 여러 개를 선택할 수 있게 하려면 Checkbox를 그대로 사용하고 선택 로직을 조정해야 합니다.
radio_buttons = widgets.RadioButtons(
    options=list(job_stability_options.keys()),
    description='직업 안정성:',
    disabled=False,
    tooltips=list(job_stability_options.values())
)

# 결과 출력 영역
output_job_stability = widgets.Output()

# 버튼
button_job_stability = widgets.Button(
    description='직업 안정성 점수 계산',
    button_style='primary'
)

def on_job_stability_button_click(b):
    with output_job_stability:
        clear_output()

        selected_job_type = radio_buttons.value

        # 함수 실행
        job_stability_category, job_stability_score = get_job_stability_from_ui(selected_job_type)

        print(f"선택한 직업 유형: {selected_job_type}")
        print(f"Category: {job_stability_category}")
        print(f"Score: {job_stability_score}")

button_job_stability.on_click(on_job_stability_button_click)

# 화면에 표시
display(widgets.VBox([radio_buttons, button_job_stability, output_job_stability]))

# 2. 나이

In [69]:
# =========================================================
# 2-1. 나이 점수
# =========================================================

def score_age(age):
    """
    나이 기반 점수화 함수

    젊을수록 투자 가능 기간과 손실 회복 기간이 길다고 보고 고점수.
    """

    if pd.isna(age):
        return np.nan

    if age < 30:
        return 10
    elif age < 40:
        return 8
    elif age < 50:
        return 6
    elif age < 60:
        return 4
    elif age < 65:
        return 2
    else:
        return 1

In [70]:
# =========================================================
# 2-2. 은퇴까지 남은 기간 점수
# =========================================================

def score_retirement_period(retirement_period_category):
    """
    은퇴까지 남은 기간 범주 점수화 함수

    UI에서는 체크박스 또는 선택지로 아래 범주 중 하나를 받는다고 가정.

    입력 예시:
    - "30년 이상"
    - "20년 이상 ~ 30년 미만"
    - "10년 이상 ~ 20년 미만"
    - "5년 이상 ~ 10년 미만"
    - "5년 미만"
    """

    retirement_score_map = {
        "30년 이상": 10,
        "20년 이상 ~ 30년 미만": 8,
        "10년 이상 ~ 20년 미만": 6,
        "5년 이상 ~ 10년 미만": 3,
        "5년 미만": 1
    }

    return retirement_score_map.get(retirement_period_category, np.nan)

In [71]:
# =========================================================
# 2-3. 시간 여유 점수
# =========================================================
import pandas as pd
import numpy as np
def calculate_time_horizon_score(age, retirement_period_category):
    """
    시간 여유 점수 계산

    시간 여유 점수 = 나이 점수 * 0.4 + 은퇴까지 남은 기간 점수 * 0.6
    """

    age_score = score_age(age)
    retirement_score = score_retirement_period(retirement_period_category)

    if pd.isna(age_score) or pd.isna(retirement_score):
        return np.nan

    time_horizon_score = age_score * 0.4 + retirement_score * 0.6
    time_horizon_score_scaled = (time_horizon_score - 1) / 9

    return time_horizon_score_scaled

In [72]:
#예시
calculate_time_horizon_score(
    age=35,
    retirement_period_category="20년 이상 ~ 30년 미만"
)

0.7777777777777778

In [73]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Age Input Widget
age_slider = widgets.IntSlider(
    value=30,
    min=18,
    max=90,
    step=1,
    description='나이:',
    continuous_update=False
)

# Retirement Period Category Radio Buttons
retirement_options = [
    "30년 이상",
    "20년 이상 ~ 30년 미만",
    "10년 이상 ~ 20년 미만",
    "5년 이상 ~ 10년 미만",
    "5년 미만"
]
retirement_radio_buttons = widgets.RadioButtons(
    options=retirement_options,
    description='은퇴까지 남은 기간:',
    disabled=False
)

# Output Area
output_time_horizon = widgets.Output()

# Button
button_time_horizon = widgets.Button(
    description='시간 여유 점수 계산',
    button_style='primary'
)

def on_time_horizon_button_click(b):
    with output_time_horizon:
        clear_output()

        selected_age = age_slider.value
        selected_retirement_period = retirement_radio_buttons.value

        # 함수 실행
        time_horizon_score = calculate_time_horizon_score(
            age=selected_age,
            retirement_period_category=selected_retirement_period
        )

        print(f"선택한 나이: {selected_age}")
        print(f"선택한 은퇴까지 남은 기간: {selected_retirement_period}")
        print(f"시간 여유 점수: {time_horizon_score}")

button_time_horizon.on_click(on_time_horizon_button_click)

# Display Widgets
display(widgets.VBox([
    age_slider,
    retirement_radio_buttons,
    button_time_horizon,
    output_time_horizon
]))

# 3. 가족

In [74]:
def calculate_family_score(selected_family):
    """
    가족 구성 체크박스 입력값을 기반으로 family_score를 계산하는 함수

    Parameters
    ----------
    selected_family : list
        UI 체크박스에서 선택된 가족 구성 리스트
        가능한 값:
        - 'alone'       : 혼자 거주
        - 'spouse'      : 배우자
        - 'children'    : 자녀
        - 'parents'     : 부모
        - 'grandparents': 조부모
        - 'siblings'    : 형제자매
        - 'relatives'   : 친인척
        - 'others'      : 기타 동거인

    Returns
    -------
    dict
        family_group, family_score, description 반환
    """

    # 아무것도 선택 안 했을 때
    if selected_family is None or len(selected_family) == 0:
        return {
            'family_group': 'unknown',
            'family_score': 0.50,
            'description': '가족 구성 정보 없음 → 중립 처리'
        }

    selected_family = set(selected_family)

    # 1. 혼자 거주
    # 혼자 거주는 다른 가족 구성과 동시에 선택되면 논리적으로 이상하므로
    # alone만 단독 선택된 경우만 인정
    if selected_family == {'alone'}:
        return {
            'family_group': 'single',
            'family_score': 1.00,
            'description': '혼자 거주 → 부양책임 가능성 낮음'
        }

    # 2. 자녀 포함
    # 자녀가 있으면 배우자 여부와 관계없이 부양책임 가능성이 높다고 판단
    if 'children' in selected_family:
        if 'spouse' in selected_family:
            return {
                'family_group': 'spouse_children',
                'family_score': 0.25,
                'description': '배우자 및 자녀와 거주 → 부양책임 가능성 높음'
            }
        else:
            return {
                'family_group': 'with_children',
                'family_score': 0.25,
                'description': '자녀와 거주 → 부양책임 가능성 높음'
            }

    # 3. 다세대 가구
    # 배우자 + 부모/조부모 등 세대가 확장된 경우
    if 'spouse' in selected_family and any(x in selected_family for x in ['parents', 'grandparents']):
        return {
            'family_group': 'multi_generation',
            'family_score': 0.25,
            'description': '배우자와 부모/조부모 등 다세대 거주 → 부양책임 가능성 높음'
        }

    # 4. 배우자만 있는 경우
    if selected_family == {'spouse'}:
        return {
            'family_group': 'couple_only',
            'family_score': 0.75,
            'description': '배우자와 거주 → 자녀 부양 부담은 낮지만 공동 재무목표 존재 가능'
        }

    # 5. 부모/조부모와 거주
    # 부양인지 지원받는 상황인지 불명확하므로 중립
    if any(x in selected_family for x in ['parents', 'grandparents']):
        return {
            'family_group': 'with_parents',
            'family_score': 0.50,
            'description': '부모 또는 조부모와 거주 → 부양/지원 여부가 불명확하여 중립 처리'
        }

    # 6. 형제자매/친인척/기타 동거
    if any(x in selected_family for x in ['siblings', 'relatives', 'others']):
        return {
            'family_group': 'others',
            'family_score': 0.50,
            'description': '기타 동거 유형 → 부양책임 해석이 불명확하여 중립 처리'
        }

    # 7. 예외 처리
    return {
        'family_group': 'unknown',
        'family_score': 0.50,
        'description': '해석 불가능한 가족 구성 → 중립 처리'
    }

참고로 최종 UI
[ ] 혼자 거주
[ ] 배우자
[ ] 자녀
[ ] 부모
[ ] 조부모
[ ] 형제자매
[ ] 친인척
[ ] 기타 동거인

In [75]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# 체크박스 옵션
family_options = {
    'alone': '혼자 거주',
    'spouse': '배우자',
    'children': '자녀',
    'parents': '부모',
    'grandparents': '조부모',
    'siblings': '형제자매',
    'relatives': '친인척',
    'others': '기타 동거인'
}

# 체크박스 생성
checkboxes = {
    key: widgets.Checkbox(
        value=False,
        description=label,
        indent=False
    )
    for key, label in family_options.items()
}

# 결과 출력 영역
output = widgets.Output()

# 버튼
button = widgets.Button(
    description='가족 점수 계산',
    button_style='primary'
)

def on_button_click(b):
    with output:
        clear_output()

        # 체크된 항목만 리스트로 저장
        selected_family = [
            key for key, checkbox in checkboxes.items()
            if checkbox.value
        ]

        # 함수 실행
        result = calculate_family_score(selected_family)

        html_output = f"""
            <div style="border: 1px solid #eee; padding: 10px; border-radius: 5px; background-color: #f9f9f9; margin-top: 10px;">
                <p><b>선택한 가족 구성:</b> {', '.join([family_options[s] for s in selected_family]) if selected_family else '없음'}</p>
                <p><b>Family Group:</b> {result['family_group']}</p>
                <p><b>Family Score:</b> <span style="color: #1a73e8; font-weight: bold;">{result['family_score']:.2f}</span></p>
                <p><b>설명:</b> {result['description']}</p>
            </div>
        """
        display(HTML(html_output))

button.on_click(on_button_click)

# 화면에 표시
display(widgets.VBox(list(checkboxes.values()) + [button, output]))

# 자금

In [76]:
import pandas as pd
import numpy as np


# =========================================================
# 최종 Risk Score 계산 함수
# =========================================================

def calculate_final_risk_score(
    selected_job_type,
    age,
    retirement_period_category,
    selected_family,
    investable_capital,
    years_worked,
    years_to_retire,
    income_dependency,
    job_type,
    expected_contribution,
    income_level
):
    """
    최종 Risk Score 계산 함수

    최종 Risk Score =
        자금 여력 점수 × 0.30
      + 시간 여력 점수 × 0.30
      + 직업 안정성 점수 × 0.25
      + 가족 점수 × 0.15

    출력 범위: 0.0 ~ 1.0
    높을수록 위험 감수 가능성이 높음
    """

    # -----------------------------------------------------
    # 1. 직업 안정성 점수
    # get_job_stability_from_ui() 결과 사용
    # -----------------------------------------------------
    job_stability_category, job_stability_score = get_job_stability_from_ui(
        selected_job_type
    )

    # -----------------------------------------------------
    # 2. 시간 여력 점수
    # calculate_time_horizon_score() 결과 사용
    # -----------------------------------------------------
    time_horizon_score = calculate_time_horizon_score(
        age,
        retirement_period_category
    )

    # -----------------------------------------------------
    # 3. 가족 부양 부담 점수
    # calculate_family_score() 결과 사용
    # -----------------------------------------------------
    family_result = calculate_family_score(selected_family)
    family_score = family_result["family_score"]

    # -----------------------------------------------------
    # 4. 자금 여력 점수
    # Part 1 + Part 2 기반 total_score 사용
    # -----------------------------------------------------

    # Part 1. Capital / Career Capacity Score
    part1_result = calculate_capital_career_capacity_score(
        investable_capital=investable_capital,
        years_worked=years_worked,
        years_to_retire=years_to_retire,
        income_dependency=income_dependency
    )

    part1_score_01 = part1_result["Part 1 점수(0~1)"]

    # Part 2. 퇴직연금 기여 안정성 Score
    part2_result = calculate_pension_contribution_stability_score(
        job_type=job_type,
        expected_contribution=expected_contribution,
        income_level=income_level
    )

    part2_score_01 = part2_result["Part 2 점수(0~1)"]

    # 자금 여력 점수
    total_score = calculate_total_score(
        part1_score_01=part1_score_01,
        part2_score_01=part2_score_01
    )

    # -----------------------------------------------------
    # 5. 최종 Risk Score
    # -----------------------------------------------------
    final_risk_score = (
        total_score * 0.30
        + time_horizon_score * 0.30
        + job_stability_score * 0.25
        + family_score * 0.15
    )

    # 0.0 ~ 1.0 범위 보정
    final_risk_score = max(0.0, min(1.0, final_risk_score))

    return {
        "job_stability_category": job_stability_category,
        "job_stability_score": round(job_stability_score, 3),
        "time_horizon_score": round(time_horizon_score, 3),
        "family_score": round(family_score, 3),
        "part1_score": round(part1_score_01, 3),
        "part2_score": round(part2_score_01, 3),
        "capital_total_score": round(total_score, 3),
        "final_risk_score": round(final_risk_score, 3)
    }

In [77]:
selected_job_type = "안정적 직업군"
age = 35
retirement_period_category = "20년 이상 ~ 30년 미만"
selected_family = ['spouse', 'children']
investable_capital = 500
years_worked = 5
years_to_retire = 25
income_dependency = "높음: 소득의 60~80%가 근로소득"
job_type = "일반 직장인"
expected_contribution = "보통: 당분간 근속 가능"
income_level = "중간"

result = calculate_final_risk_score(
    selected_job_type=selected_job_type,
    age=age,
    retirement_period_category=retirement_period_category,
    selected_family=selected_family,
    investable_capital=investable_capital,
    years_worked=years_worked,
    years_to_retire=years_to_retire,
    income_dependency=income_dependency,
    job_type=job_type,
    expected_contribution=expected_contribution,
    income_level=income_level
)

print("========== 최종 Risk Score 계산 결과 ==========")
print(f"직업 안정성 점수: {result['job_stability_score']}")
print(f"시간 여력 점수: {result['time_horizon_score']}")
print(f"가족 점수: {result['family_score']}")
print(f"Part 1 점수: {result['part1_score']}")
print(f"Part 2 점수: {result['part2_score']}")
print(f"자금 여력 점수: {result['capital_total_score']}")
print("---------------------------------------------")
print(f"최종 Risk Score: {result['final_risk_score']}")

========== 최종 Risk Score 계산 결과 ==========
직업 안정성 점수: 1.0
시간 여력 점수: 0.778
가족 점수: 0.25
Part 1 점수: 0.54
Part 2 점수: 0.68
자금 여력 점수: 0.61
---------------------------------------------
최종 Risk Score: 0.704


In [78]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Assuming the scores have been calculated and are available in the kernel
# (e.g., from the previous cell c2eb4fd0's execution)
# total_score (자금 여력 점수)
# time_horizon_score (시간 여력 점수)
# job_stability_score (직업 안정성 점수)
# family_score (가족 부양 부담 점수)
# final_risk_score (최종 Risk Score)

# Create a styled HTML output for better readability
html_output = f"""
    <div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px; font-family: sans-serif;">
        <h3 style="color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;">최종 Risk Score 결과</h3>
        <p style="font-size: 1.1em;"><b>자금 여력 점수:</b> <span style="color: #27ae60;">{total_score:.3f}</span></p>
        <p style="font-size: 1.1em;"><b>시간 여력 점수:</b> <span style="color: #27ae60;">{time_horizon_score:.3f}</span></p>
        <p style="font-size: 1.1em;"><b>직업 안정성 점수:</b> <span style="color: #27ae60;">{job_stability_score:.3f}</span></p>
        <p style="font-size: 1.1em;"><b>가족 부양 부담 점수:</b> <span style="color: #27ae60;">{family_score:.3f}</span></p>
        <h3 style="color: #e74c3c; margin-top: 15px;">최종 Risk Score: <span style="font-size: 1.3em;">{final_risk_score:.3f}</span></h3>
    </div>
"""

display(HTML(html_output))


# 최종 Risk Score 통합 UI

In [79]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# =========================================================
# 1. Input Widgets (재사용 및 통합)
# =========================================================

# 1.1 직업 안정성 점수 (get_job_stability_from_ui 용 selected_job_type)
job_stability_options_final_ui = {
    "안정적 직업군": "안정적 직업군",
    "일반 직장인·기술직": "일반 직장인·기술직",
    "소득 변동 가능 직업군": "소득 변동 가능 직업군"
}
job_stability_radio_buttons_final_ui = widgets.RadioButtons(
    options=list(job_stability_options_final_ui.keys()),
    description='직업 안정성 (주요 직업군):',
    disabled=False,
    tooltips=list(job_stability_options_final_ui.values()),
    value="안정적 직업군" # Default value
)

# 1.2 시간 여력 점수 (age, retirement_period_category)
age_slider_final_ui = widgets.IntSlider(
    value=35,
    min=18,
    max=90,
    step=1,
    description='나이:',
    continuous_update=False,
    style={'description_width': 'initial'}
)
retirement_options_final_ui = [
    "30년 이상",
    "20년 이상 ~ 30년 미만",
    "10년 이상 ~ 20년 미만",
    "5년 이상 ~ 10년 미만",
    "5년 미만"
]
retirement_radio_buttons_final_ui = widgets.RadioButtons(
    options=retirement_options_final_ui,
    description='은퇴까지 남은 기간:',
    disabled=False,
    value="20년 이상 ~ 30년 미만" # Default value
)

# 1.3 가족 부양 부담 점수 (selected_family)
family_options_final_ui = {
    'alone': '혼자 거주',
    'spouse': '배우자',
    'children': '자녀',
    'parents': '부모',
    'grandparents': '조부모',
    'siblings': '형제자매',
    'relatives': '친인척',
    'others': '기타 동거인'
}
checkboxes_final_ui = {
    key: widgets.Checkbox(
        value=(key in ['spouse', 'children']), # Default selection for demonstration
        description=label,
        indent=False
    )
    for key, label in family_options_final_ui.items()
}
family_checkbox_vbox_final_ui = widgets.VBox(list(checkboxes_final_ui.values()), layout=widgets.Layout(border='solid 1px lightgray', padding='10px'))


# 1.4 자금 여력 점수 (investable_capital, years_worked, years_to_retire, income_dependency, job_type, expected_contribution, income_level)
investable_capital_widget_final_ui = widgets.IntSlider(
    value=500,
    min=0,
    max=10000,
    step=100,
    description="투자자금(만원)",
    style={'description_width': 'initial'}
)
years_worked_widget_final_ui = widgets.IntSlider(
    value=5,
    min=0,
    max=40,
    step=1,
    description="현재까지 재직기간 (년)",
    style={'description_width': 'initial'}
)
years_to_retire_widget_final_ui = widgets.IntSlider(
    value=25,
    min=0,
    max=45,
    step=1,
    description="은퇴까지 남은 기간 (년)",
    style={'description_width': 'initial'}
)
income_dependency_widget_final_ui = widgets.RadioButtons(
    options=[
        "매우 높음: 소득의 80% 이상이 근로소득",
        "높음: 소득의 60~80%가 근로소득",
        "보통: 소득의 40~60%가 근로소득",
        "낮음: 소득의 40% 미만이 근로소득"
    ],
    value="높음: 소득의 60~80%가 근로소득",
    description="근로소득 의존도",
    style={'description_width': 'initial'}
)
job_type_widget_final_ui = widgets.RadioButtons(
    options=[
        "고정 소득 기반 직업군",
        "일반 직장인",
        "성과/계약 변동 직업군",
        "무직/구직중"
    ],
    value="일반 직장인",
    description="직업 안정성 (자금 여력용):",
    style={'description_width': 'initial'}
)
expected_contribution_widget_final_ui = widgets.RadioButtons(
    options=[
        "높음: 장기 근속 가능성이 높음",
        "보통: 당분간 근속 가능",
        "낮음: 이직/퇴직 가능성 있음",
        "매우 낮음: 근속 불확실성이 큼"
    ],
    value="보통: 당분간 근속 가능",
    description="예상 근속 가능성",
    style={'description_width': 'initial'}
)
income_level_widget_final_ui = widgets.RadioButtons(
    options=[
        "낮음",
        "중간",
        "높음",
        "매우 높음"
    ],
    value="중간",
    description="소득 수준",
    style={'description_width': 'initial'}
)

# =========================================================
# 2. Button and Output Area
# =========================================================

calculate_final_score_button = widgets.Button(
    description="최종 Risk Score 계산하기",
    button_style="success",
    layout=widgets.Layout(width="250px", height="45px", margin='20px 0 0 0')
)

final_score_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='15px',
        margin='10px 0',
        min_height='200px'
    )
)

# =========================================================
# 3. Button Click Event Handler
# =========================================================

def on_calculate_final_score_clicked(b):
    with final_score_output:
        clear_output()

        # Gather all inputs from the widgets
        selected_job_type_val = job_stability_radio_buttons_final_ui.value
        age_val = age_slider_final_ui.value
        retirement_period_category_val = retirement_radio_buttons_final_ui.value
        selected_family_val = [key for key, checkbox in checkboxes_final_ui.items() if checkbox.value]
        investable_capital_val = investable_capital_widget_final_ui.value
        years_worked_val = years_worked_widget_final_ui.value
        years_to_retire_val = years_to_retire_widget_final_ui.value
        income_dependency_val = income_dependency_widget_final_ui.value
        job_type_val = job_type_widget_final_ui.value
        expected_contribution_val = expected_contribution_widget_final_ui.value
        income_level_val = income_level_widget_final_ui.value

        # Call the main calculation function
        try:
            result = calculate_final_risk_score(
                selected_job_type=selected_job_type_val,
                age=age_val,
                retirement_period_category=retirement_period_category_val,
                selected_family=selected_family_val,
                investable_capital=investable_capital_val,
                years_worked=years_worked_val,
                years_to_retire=years_to_retire_val,
                income_dependency=income_dependency_val,
                job_type=job_type_val,
                expected_contribution=expected_contribution_val,
                income_level=income_level_val
            )

            # Display the results in a formatted HTML structure similar to the example
            html_result = f"""
                <div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px; font-family: sans-serif;">
                    <h3 style="color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;">최종 Risk Score 결과</h3>
                    <p style="font-size: 1.1em;"><b>직업 안정성 점수:</b> <span style="color: #27ae60;">{result['job_stability_score']:.3f}</span></p>
                    <p style="font-size: 1.1em;"><b>시간 여력 점수:</b> <span style="color: #27ae60;">{result['time_horizon_score']:.3f}</span></p>
                    <p style="font-size: 1.1em;"><b>가족 부양 부담 점수:</b> <span style="color: #27ae60;">{result['family_score']:.3f}</span></p>
                    <p style="font-size: 1.1em;"><b>자금 여력 점수:</b> <span style="color: #27ae60;">{result['capital_total_score']:.3f}</span></p>
                    <h3 style="color: #e74c3c; margin-top: 15px;">최종 Risk Score: <span style="font-size: 1.3em;">{result['final_risk_score']:.3f}</span></h3>
                </div>
            """
            display(HTML(html_result))
        except NameError as e:
            display(HTML(f"<p style='color:red;'><b>오류 발생:</b> {e}<br>일부 함수가 정의되지 않았을 수 있습니다. 상위 셀들을 먼저 실행해주세요.</p>"))
        except Exception as e:
            display(HTML(f"<p style='color:red;'><b>예상치 못한 오류 발생:</b> {e}</p>"))

calculate_final_score_button.on_click(on_calculate_final_score_clicked)

# =========================================================
# 4. Display the Consolidated UI
# =========================================================

display(
    widgets.VBox([
        widgets.HTML("<h2>종합 Risk Score 계산기</h2><hr>"),

        widgets.HTML("<h3>1. 직업 안정성</h3>"),
        job_stability_radio_buttons_final_ui,

        widgets.HTML("<h3>2. 시간 여력</h3>"),
        age_slider_final_ui,
        retirement_radio_buttons_final_ui,

        widgets.HTML("<h3>3. 가족 부양 부담</h3>"),
        widgets.Label("가족 구성 선택:"),
        family_checkbox_vbox_final_ui,

        widgets.HTML("<h3>4. 자금 여력 (Part 1 & 2)</h3>"),
        investable_capital_widget_final_ui,
        years_worked_widget_final_ui,
        years_to_retire_widget_final_ui,
        income_dependency_widget_final_ui,
        job_type_widget_final_ui,
        expected_contribution_widget_final_ui,
        income_level_widget_final_ui,

        calculate_final_score_button,
        final_score_output
    ],
    layout=widgets.Layout(
        border='2px solid #a0a0a0',
        padding='20px',
        margin='20px 0',
        width='900px'
    ))
)